# 네이버 증권 뉴스 탭의 3일치 기사를 수집
데이터 프레임으로 생성하고 파일롤 저장
사이트 주소: https://finance.naver.com/news/mainnews.naver?date=2024-10-25

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

url = 'https://finance.naver.com/news/mainnews.naver?date=2026-02-20'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'lxml')

In [ ]:
# 기사제목 리스트 가져오기
titles = soup.find_all('dd', {'class':'articleSubject'})
print(titles[:2])

# 첫번째 기사의 기사 제목 
print(titles[0].text.strip())

In [ ]:
# 첫번째 기사의 상세페이지 URL

titles[0].find('a').get('href')
news_url = 'https://finance.naver.com' + titles[0].find('a').get('href')

# 상세 뉴스페이지 가져오기
news_response = requests.get(news_url)
news_soup = BeautifulSoup(news_response.text, 'lxml')
print(news_soup)

In [ ]:
#redirect 주소를 반환하고 있으므로, redirecet 주소를 사용하여 다시 상세 뉴스 페이지에 접근
news_url2 = news_soup.find('script').text.split("'")[1]

# 상세 뉴스페이지 재도전
news_response2 = requests.get(news_url2)
news_soup = BeautifulSoup(news_response2.text, 'lxml')
print(news_soup.prettify()[:1000])

# 상세 뉴스 페이지 본문 파싱하기

In [ ]:
# 뉴스 본문
# 기사 본문: <div id="newsct_article" class="newsct_article _article_body">
news_1 = news_soup.find('div', {'class':'newsct_article _article_body'}).text.strip()
print(news_1)

# 함수로 만들기

In [ ]:
#반복문 사용하여 함수로 생성하기
import datetime
import time

def get_news_items(html):
  # 기사제목 리스트 가져오기
  titles = html.find_all('dd', {'class':'articleSubject'})
  title_list = []
  url_list = []
  article_list = []

  for t in titles:
    # 기사의 제목
    title = t.text.strip()

    # 기사의 상세페이지 URL
    news_url = 'https://finance.naver.com' + t.find('a').get('href')

    # 상세페이지 request
    news_response = requests.get(news_url)
    news_soup = BeautifulSoup(news_response.text, 'lxml')

    # 리다이렉트 주소를 파싱
    news_url2 = news_soup.find('script').text.split("'")[1]

    # 상세 뉴스페이지 request 재도전
    news_response2 = requests.get(news_url2)
    news_soup = BeautifulSoup(news_response2.text, 'lxml')

    # 신문기사 본문 파싱
    article = news_soup.find('div', {'class':'newsct_article _article_body'}).text.strip()

    # 리스트에 값 채우기
    title_list.append(title)
    url_list.append(news_url2)
    article_list.append(article)

  df = pd.DataFrame({'기사제목': title_list, '본문url': url_list, '기사본문': article_list})
  return df

In [ ]:
news_df = pd.DataFrame()

for i in range(3):
    # 오늘 날짜 기준 i일 이전 날짜 구하기
    date= datetime.date.today() - datetime.timedelta(days = i)
    url = f'https://finance.naver.com/news/mainnews.naver?date={date}'
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'lxml')
    time.sleep(1)

    # 해당일 신문기사 스크랩 데이터프레임 반환
    temp_df = get_news_items(soup)
    temp_df['date'] = date
    news_df = pd.concat([news_df, temp_df], ignore_index=True).copy()

    print(news_df.head())
    news_df.to_csv('article.csv')